# AgentBase Bash 工具调用流程测试

测试流程：
1. 初始化基础设施（EventBus、ToolEventFactory、LLM Client）
2. 注册 Bash 工具及其事件处理器
3. 创建 AgentBase 实例，注入上下文引擎
4. 提交任务，观察 think → tool_call → event → handler → callback 完整链路

In [21]:
import asyncio
import os
from dotenv import load_dotenv

load_dotenv()

True

## 1. 初始化基础设施

In [22]:
from infra.eventbus import EventBus
from domain.event import ToolEventFactory
from infra.tool.builtin.declare import *
from domain.agent.write.tools import *
from infra.LLM.LLM_infra import LLM_Client, LLM_Model_Provider

# 事件总线
bus = EventBus()
# 工具工厂（注册已有 Tool 实例，构建事件描述符）
factory = ToolEventFactory(prefix="infra")._build()._resigister_bus(bus)
# LLM 客户端
llm_client = LLM_Client(
    url=os.getenv("LLM_BASE_URL", "https://api.minimaxi.com/v1"),
    model_class="MiniMax-M2.7",
    model_provider=LLM_Model_Provider.MINMAX,
    max_tokens=131072,
)

## 2. 注册 Bash 工具 & 事件处理器

- `infra/tool/builtin/system.py` 会注册 BASH 工具到 factory，并绑定 `bash.called` 事件处理器
- `infra/tool/tools_attach_methods.py` 会注册通用 `*.succeeded` / `*.failed` 模式匹配处理器

In [23]:

# 导入即注册：
#   - system.py 在模块级别执行 factory._build_and_register_list([BASH], bus)
#   - 并通过 @on_tool.on(factory.tool("bash").called()) 绑定 exec_bash 处理器
from infra.tool.builtin.system import BASH, SystemTool

# 导入即注册：
#   - logging 中间件
#   - *.succeeded / *.failed 模式匹配回调
from infra.tool.tools_attach_methods import *

In [24]:
# 验证：bash 工具已注册到 factory
print("bash 事件名:", factory.tool("bash").all_event_names())
print("所有已注册工具:", list(factory._specs.keys()))

bash 事件名: ['infra.system.bash.called', 'infra.system.bash.failed', 'infra.system.bash.retrying', 'infra.system.bash.succeeded']
所有已注册工具: ['write_plan', 'update_plan', 'finish_plan', 'confirm_human', 'requirements_analysis', 'outline_generation', 'draft_writing', 'rewrite', 'style_polish', 'requirements_check', 'bash']


## 3. 创建 AgentBase 实例

AgentBase 是抽象基类，这里创建一个最小子类用于测试

In [25]:
from domain.agent_base import AgentBase
from domain.context.context import ContextEngine
from domain.context.providers import (
    UserPromptProvider,
    StateProvider,
    AvailableToolsProvider,
    HistoryProvider,
    ToolOutputProvider,
)
from domain.context.strategy import FullHistoryStrategy, RecencyStrategy, TokenBudgetStrategy
from domain.memory.short.default_short_term_memory import DefaultShortTermMemory


class TestBashAgent(AgentBase):
    """用于测试 Bash 工具调用的最小 Agent 子类"""

    def _build_agent_prompt(self) -> str:
        return f"""
你是一个系统操作助手，当前工作目录为：{self.work_path}
你可以使用 bash 工具执行命令来完成任务。

## 输出格式
用 JSON 严格按以下格式回复：
{{
  "think": "你的思考过程",
  "tool_calls": [
    {{
      "tool_name": "bash",
      "arguments": {{"command": "要执行的命令"}},
      "reasoning": "为什么执行这个命令"
    }}
  ],
  "is_finished": false
}}

## 任务完成时输出
{{
  "think": "...",
  "tool_calls": [],
  "is_finished": true,
  "finish_reason": "完成原因"
}}
"""

In [26]:
# 构建上下文引擎（精简版：只注入 bash 所需的 provider）
from infra.context.db_strategy import ChunkToFileStrategy


memory = DefaultShortTermMemory(["tool_respond", "agent_history"])

providers = [
    UserPromptProvider(),
    StateProvider(),
    AvailableToolsProvider(["system"]),  # 只暴露 system 组工具（bash）
    HistoryProvider(memory, "agent_history", FullHistoryStrategy()),
    ToolOutputProvider(memory, "tool_respond", FullHistoryStrategy() | RecencyStrategy(10)),
]

context_engine = ContextEngine(
    providers=providers,
    memory=memory,
)

In [27]:
# 创建 Agent 实例
agent = TestBashAgent(
    id="bash_test_001",
    name="BashTester",
    llm=llm_client,
    context=context_engine,
)

print(f"Agent ID: {agent.id}")
print(f"Agent Name: {agent.name}")
print(f"工作目录: {agent.work_path}")

Agent ID: bash_test_001
Agent Name: BashTester
工作目录: /Users/zxcvbzzy1/Desktop/项目/agent_flow/temp


## 4. 运行测试

In [28]:
def run_async(coro):
    """兼容 Jupyter 已有事件循环的场景"""
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    return loop.create_task(coro)

In [29]:
async def run_bash_test():
    """
    Bash 工具调用流程完整测试

    预期事件流：
    1. Agent _think() → LLM 返回 tool_calls: [{tool_name: "bash", arguments: {command: "..."}}]
    2. _execute_tools() → factory.tool("bash").emit_called() → 发布 infra.system.bash.called
    3. exec_bash 处理器 → 安全审核 → subprocess.run → 返回 succeeded/failed 事件
    4. *.succeeded/*.failed 模式处理器 → agent.on_tool_call() → 更新 state & memory
    5. _tool_done.set() → Agent 进入下一轮 _think()
    """
    print("=" * 60)
    print("AgentBase Bash 工具调用流程测试")
    print("=" * 60)

    test_prompt = """请帮我完成以下操作：
1. 列出当前工作目录下的文件和文件夹
2. 查看当前目录下是否有 .md 文件
3. 如果有 .md 文件，显示它们的内容
4. 如果是文件内容是英文，翻译为中文，并写个md文件存储
5. 通过python脚本处理将这些文件转为pdf，如果不能，则用python对文件进行切片，每片1000字，
存储为新的md文件，其中python环境为/Users/zxcvbzzy1/miniconda3/envs/MY_env/bin
6. 完成后汇报结果。
"""

    print(f"\n测试任务:\n{test_prompt}")
    print("\n开始执行...\n")
    print("=" * 60)

    try:
        await agent.start(prompt=test_prompt)

        print("\n" + "=" * 60)
        print("Agent 执行完成！")
        print("=" * 60)

        # 查看最终状态
        final_state = agent.states_manage.get_state()
        print(f"\n是否完成: {final_state.get('is_finished', False)}")
        print(f"完成原因: {final_state.get('finish_reason', 'N/A')}")
        print(f"思考过程: {final_state.get('think', 'N/A')[:500]}...")
        print(f"工具调用历史: {' -> '.join(final_state.get('tool_history', []))}")
        print(f"重试次数: {final_state.get('retry', 0)}")

        # 查看 memory 中的工具输出
        mem = agent.context_engine.get_memory()
        tool_keys = mem.keys_by_field("tool_respond")
        print(f"\nMemory 中工具输出条目: {len(tool_keys)}")
        for key in tool_keys:
            result = mem.get("tool_respond", key)
            print(f"  [{key}] {result[:200]}..." if len(result) > 200 else f"  [{key}] {result}")

        return agent

    except Exception as e:
        print(f"\n执行出错: {str(e)}")
        import traceback
        traceback.print_exc()

In [ ]:
# 完整运行测试
await run_bash_test()

## 5. 单独测试 Bash 安全审核（不需要 LLM）

In [ ]:
# 直接测试 SystemTool 的安全审核逻辑
tool = SystemTool(working_directory="/tmp/safe_dir")

# 正常命令
ok, reason = tool.audit_bash("ls -la")
print(f"[正常] ls -la → allowed={ok}, reason='{reason}'")

# 高危命令
ok, reason = tool.audit_bash("rm -rf /tmp/test")
print(f"[高危] rm -rf /tmp/test → allowed={ok}, reason='{reason}'")

# 路径越界
ok, reason = tool.audit_bash("cat /etc/passwd")
print(f"[越界] cat /etc/passwd → allowed={ok}, reason='{reason}'")

# Shell 嵌套高危
ok, reason = tool.audit_bash("bash -c 'kill -9 1'")
print(f"[嵌套高危] bash -c 'kill -9 1' → allowed={ok}, reason='{reason}'")

# 重定向越界
ok, reason = tool.audit_bash("echo test > /etc/test.txt")
print(f"[重定向越界] echo test > /etc/test.txt → allowed={ok}, reason='{reason}'")

In [ ]:
# 直接测试命令执行
result = tool.exec_bash("echo 'Hello from AgentBase Bash test'")
print(f"stdout: {result['stdout']}")
print(f"returncode: {result['returncode']}")

## 6. 单独测试事件链路（不经过 LLM）

手动发射事件，验证 EventBus → handler → callback 完整链路

In [ ]:
async def test_event_chain():
    """
    手动测试事件链路，绕过 LLM 决策环节：
    emit_called → exec_bash handler → succeeded/failed → on_tool_call callback
    """
    print("=" * 60)
    print("事件链路手动测试")
    print("=" * 60)

    # 1. 手动发射 bash.called 事件
    print("\n[1] 发射 infra.system.bash.called 事件...")
    await factory.tool("bash").emit_called({
        "command": "ls -la",
        "agent_id": agent.id,
    })

    # 2. 检查 agent 状态是否被回调更新
    print(f"\n[2] 回调后 Agent 状态:")
    print(f"   tool_history: {agent.states.get('tool_history', [])}")
    print(f"   last_tool_ok: {agent.states.get('last_tool_ok')}")

    # 3. 检查 memory
    mem = agent.context_engine.get_memory()
    bash_results = mem.get("tool_respond", "bash")
    print(f"\n[3] Memory 中 bash 输出:")
    print(f"   {bash_results[:300]}..." if len(bash_results) > 300 else f"   {bash_results}")

    # 4. 测试失败场景：执行一条会失败的命令
    print("\n[4] 测试失败场景：执行不存在的命令...")
    agent.states["retry"] = 0  # 重置
    await factory.tool("bash").emit_called({
        "command": "ls /nonexistent_dir_xyz",
        "agent_id": agent.id,
    })
    print(f"   last_tool_ok: {agent.states.get('last_tool_ok')}")
    print(f"   retry: {agent.states.get('retry')}")

    # 5. 测试高危命令拦截
    print("\n[5] 测试高危命令拦截...")
    agent.states["retry"] = 0
    await factory.tool("bash").emit_called({
        "command": "rm -rf /tmp/test",
        "agent_id": agent.id,
    })
    print(f"   last_tool_ok: {agent.states.get('last_tool_ok')}")
    print(f"   retry: {agent.states.get('retry')}")

    print("\n" + "=" * 60)
    print("事件链路测试完成")
    print("=" * 60)

In [ ]:
await test_event_chain()